# Preprocessing script after running simulation

## Transfer csv to npz feature values per Q and weir

In [1]:
import os
import trimesh
import pandas as pd
import numpy as np
import glob

In [ ]:
# Set your folder path here
# Path where all .csv files are stored
folder_path = "../Data/PKW_Efficiency_Dataset"
# Path where the new generated .npz file should be stored
output_npz_folder_path = "../Data/PKW_Efficiency_Dataset"

# Check if folder for .npz exists else create
os.makedirs(output_npz_folder_path, exist_ok=True)

# List all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
# csv_files = glob.glob(os.path.join(folder_path, "**", "*.csv"), recursive=True)

# Read and compare headers, so if in all .csv files the headers are the same we can continue
expected_header = None
header_mismatch = False
dataframes = []

for file in csv_files:
    file_path = os.path.join(folder_path, file)
    with open(file_path, 'r', encoding='utf-8') as f:
        header = f.readline().strip().split(',')
        
    if expected_header is None:
        expected_header = header
    elif header != expected_header:
        header_mismatch = True
        print(f"Header mismatch in file: {file}")
        break

# If all headers match, read and combine the files
if not header_mismatch:
    for file in csv_files:
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        dataframes.append(df)
    
    combined_df = pd.concat(dataframes, ignore_index=True)

    print("Combined DataFrame:")
    #print(combined_df)
    
    # ✅ Save to .npz file
    data_np = combined_df.to_numpy()
    columns_np = combined_df.columns.to_numpy()
    output_path = os.path.join(output_npz_folder_path, 'combined_data.npz')
    np.savez(output_path, data=data_np, columns=columns_np)

    print(f"\n✅ Saved combined data to: {output_path}")
    
else:
    print("Not all CSV files have the same header.")


Combined DataFrame:

✅ Saved combined data to: ../Data/PKW_OOD/combined_data.npz


## Transfer of each weir stl to npz pointcloud

In [2]:
def stl_to_pointcloud(path, n_points=1024, return_both=False):
    '''
    This method has the option to return both, normalized and non normalized data.
    '''
    
    # Load .stl mesh
    mesh = trimesh.load_mesh(path)

    # Check watertightness
    if not mesh.is_watertight:
        print(f"⚠ Not watertight: {os.path.basename(path)} → using convex hull")
        mesh = mesh.convex_hull

    # Sample surface
    points, _ = trimesh.sample.sample_surface(mesh, n_points)
    points = points.astype(np.float32)

    # Save raw before normalization
    if return_both:
        points_raw = points.copy()

    # Normalize
    centroid = np.mean(points, axis=0)
    points -= centroid  # center
    scale = np.max(np.linalg.norm(points, axis=1))  # scale to unit sphere
    points /= scale

    if return_both:
        return points_raw, points
    else:
        return points


In [3]:
# Path where all stl files are stored
stl_dir = "../Data/PKW_Efficiency_Dataset/stl"
# Path where all .npz files should be stored after generation, folder does not need to exist will be generated
pc_dir = "../Data/PKW_Efficiency_Dataset/pc"

# Check if folder for pointclouds exists else create
os.makedirs(pc_dir, exist_ok=True)

# Process each STL file
for file in os.listdir(stl_dir):
    if file.endswith('.stl'):
        try:
            filename_no_ext = os.path.splitext(file)[0]  # e.g., 'part_0123'
            
            # Extract Model ID
            model_id = filename_no_ext.replace("part_", "")
            
            # Process STL to point clouds
            raw, norm = stl_to_pointcloud(os.path.join(stl_dir, file), n_points=100_000, return_both=True)

            # Save to .npz including Modell
            save_path = os.path.join(pc_dir, f"part_{model_id}.npz")
            np.savez(save_path, original_pcd=raw, normalized_pcd=norm, model=model_id)

            print(f"✅ Saved: {model_id} → {save_path}")
        except Exception as e:
            print(f"❌ Failed on {file}: {e}")

✅ Saved: 03813 → ../Data/PKW_Efficiency_Dataset/pc/part_03813.npz
✅ Saved: 00143 → ../Data/PKW_Efficiency_Dataset/pc/part_00143.npz
✅ Saved: 03701 → ../Data/PKW_Efficiency_Dataset/pc/part_03701.npz
✅ Saved: 01623 → ../Data/PKW_Efficiency_Dataset/pc/part_01623.npz
✅ Saved: 02143 → ../Data/PKW_Efficiency_Dataset/pc/part_02143.npz
✅ Saved: 01357 → ../Data/PKW_Efficiency_Dataset/pc/part_01357.npz
✅ Saved: 02155 → ../Data/PKW_Efficiency_Dataset/pc/part_02155.npz
✅ Saved: 02078 → ../Data/PKW_Efficiency_Dataset/pc/part_02078.npz
✅ Saved: 03686 → ../Data/PKW_Efficiency_Dataset/pc/part_03686.npz
✅ Saved: 03685 → ../Data/PKW_Efficiency_Dataset/pc/part_03685.npz
✅ Saved: 02337 → ../Data/PKW_Efficiency_Dataset/pc/part_02337.npz
✅ Saved: 01073 → ../Data/PKW_Efficiency_Dataset/pc/part_01073.npz
✅ Saved: 02318 → ../Data/PKW_Efficiency_Dataset/pc/part_02318.npz
✅ Saved: 01134 → ../Data/PKW_Efficiency_Dataset/pc/part_01134.npz
✅ Saved: 00882 → ../Data/PKW_Efficiency_Dataset/pc/part_00882.npz
✅ Saved: 0

In [5]:
loaded = np.load("../Data/PKW_Efficiency_Dataset/pc/part_00001.npz", allow_pickle=True)
print(loaded)
print(loaded['original_pcd'].shape)

NpzFile '../Data/PKW_Efficiency_Dataset/pc/part_00001.npz' with keys: original_pcd, normalized_pcd, model
(100000, 3)


## Definition of Dataset as well as Dataloader

In [6]:
import numpy as np
import pandas as pd
import os

import torch
from torch.utils.data import Dataset, DataLoader

class RowDataset(Dataset):
    def __init__(self, npz_path, point_path):
        # Load the .npz
        loaded = np.load(npz_path, allow_pickle=True)
        data = loaded['data']      # shape: (rows, columns)
        columns = loaded['columns']
        
        # Restore as DataFrame for convenience
        self.df = pd.DataFrame(data, columns=columns)
        self.path_to_pointclouds = point_path
    
    def __len__(self):
        return len(self.df)  # Number of rows (samples)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        print(row)
        
        model_value = int(row['Modell'])
        
        if len(str(abs(model_value))) < 5:
            name_model_value = str(model_value).zfill(5)
        
        path_pointcloud = os.path.join(self.path_to_pointclouds,f"part_{name_model_value}.npz")
        pc_loaded = np.load(path_pointcloud, allow_pickle=True)
        pc_array = pc_loaded['normalized_pcd']
        pc_loaded = torch.tensor(pc_array, dtype=torch.float32)
        
        # Convert to tensor (optional: handle numeric/categorical separately)
        row_tensor = torch.tensor(row.values, dtype=torch.float32)
        return row_tensor,pc_loaded

In [7]:
# Path to the saved .npz file
# Path to the npz with all features from the .csv files
npz_path = "../Data/PKW_Efficiency_Dataset/combined_data.npz"
# Path to all .npz with stored pointclouds
pc_dir = "../Data/PKW_Efficiency_Dataset/pc"

# Create dataset and dataloader (row-wise!)
dataset = RowDataset(npz_path, pc_dir)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# Example usage
for row_batch, pc_batch in dataloader:
    print("Row tensor batch shape:", row_batch.shape)
    print("pc_batch shape:", pc_batch.shape)
    break

Modell          1.000000
Q             160.000000
Pressure     1615.745644
h_O           427.203939
h_t            97.203939
H_t            97.203939
T_s             5.000000
B_b           225.643310
N_B_i           0.625311
N_B_o           0.910186
W_i_u          75.150373
W_i_d          44.541906
Cycles          3.000000
B_i           141.097275
B_o           205.377281
B             572.117866
Alpha_rad       0.026979
Alpha_deg       1.545808
Ts_stern        5.001820
Ts_dach         4.866890
W_o_u         248.449179
W_o_d         279.057646
W_u           333.333333
W            1000.000000
P             330.000000
L            4312.930148
Ht_P            0.294557
C_d             0.414537
Name: 0, dtype: float64
Modell          2.000000
Q             160.000000
Pressure     2056.906535
h_O           472.174468
h_t           142.174468
H_t           142.174468
T_s             5.000000
B_b           480.853688
N_B_i           0.415795
N_B_o           0.401002
W_i_u         310.568641
W